In [ ]:
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# Đảm bảo thư mục tồn tại
os.makedirs("../models", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

SEED = 42
\n

In [ ]:
print("--- 1. ĐỌC DỮ LIỆU ---")
train = pd.read_csv("../data/processed/03_train.csv")
val = pd.read_csv("../data/processed/03_val.csv")
test = pd.read_csv("../data/processed/03_test.csv")

# Xóa missing value nếu có sau khi đọc
train = train.dropna(subset=['text_seg'])
val = val.dropna(subset=['text_seg'])
test = test.dropna(subset=['text_seg'])

X_train, y_train = train['text_seg'], train['label']
X_val, y_val = val['text_seg'], val['label']
X_test, y_test = test['text_seg'], test['label']

print(f"Train size: {len(X_train)} | Val size: {len(X_val)}")
\n

In [ ]:
print("--- 2. VECTOR HÓA & 3. HUẤN LUYỆN (LOGISTIC REGRESSION & LINEARSVC) ---")
# Khởi tạo vectorizer: kết hợp unigram, bigram (word) và n-gram ký tự (char_wb 2-5)
tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9
)
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    min_df=5,
    max_df=0.9
)

from sklearn.pipeline import FeatureUnion

# Kết hợp Word và Char n-grams
combined_features = FeatureUnion([
    ("word_tfidf", tfidf),
    ("char_tfidf", tfidf_char)
])

# Định nghĩa các mô hình thử nghiệm
models = {
    "LogisticRegression_C1": LogisticRegression(C=1.0, class_weight="balanced", random_state=SEED, max_iter=1000),
    "LogisticRegression_C0.1": LogisticRegression(C=0.1, class_weight="balanced", random_state=SEED, max_iter=1000),
    "LinearSVC_C1": LinearSVC(C=1.0, class_weight="balanced", random_state=SEED, max_iter=2000),
    "LinearSVC_C0.1": LinearSVC(C=0.1, class_weight="balanced", random_state=SEED, max_iter=2000),
}

best_f1 = 0
best_model_name = ""
best_model = None
val_predictions = None

print("Đang huấn luyện và đánh giá trên tập Validation...")
for name, clf in models.items():
    pipeline = Pipeline([
        ('tfidf', combined_features),
        ('clf', clf)
    ])
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict on Validation (CHỈ đánh giá trên Val)
    y_pred_val = pipeline.predict(X_val)
    f1_macro = f1_score(y_val, y_pred_val, average="macro")
    
    print(f"[{name}] F1-macro (Val): {f1_macro:.4f}")
    
    # 4. Chọn mô hình tốt nhất
    if f1_macro > best_f1:
        best_f1 = f1_macro
        best_model_name = name
        best_model = pipeline
        val_predictions = y_pred_val

print(f"\n=> Mô hình tốt nhất: {best_model_name} với F1-macro = {best_f1:.4f}")
\n

In [ ]:
print("--- 5. BÁO CÁO MÔ HÌNH TỐT NHẤT TRÊN TẬP VAL ---")
# In classification report
target_names = ["CLEAN (0)", "OFFENSIVE (1)", "HATE (2)"]
print(classification_report(y_val, val_predictions, target_names=target_names))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_val, val_predictions)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title(f'Confusion Matrix (Val) - {best_model_name}')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.tight_layout()

cm_path = f"../outputs/confusion_matrix_{best_model_name}_val.png"
plt.savefig(cm_path, dpi=300)
print(f"Đã lưu ảnh Confusion Matrix vào: {cm_path}")
plt.show()

print("\n--- 6. LƯU MÔ HÌNH ---")
# Lưu lại bằng pickle
model_path = f"../models/baseline_{best_model_name}.pkl"
with open(model_path, "wb") as f:
    pickle.dump(best_model, f)
print(f"Chưa chạy tập test. Đã lưu mô hình tốt nhất vào: {model_path}")
print(f"Random state được sử dụng: {SEED}")
\n